# QM-augmented property predictor — quickstart

Colab-ready walkthrough of the whole pipeline on a small subset.
Ch references are to 파이썬을 이용한 화학 인공지능 (정근홍, 2024).

For real runs use the scripts (`make data features train ablation`);
this notebook is for seeing each step's output.


## 0. Environment (Colab only — skip locally)


In [ ]:
# !pip install -q rdkit pyscf scikit-learn xgboost matplotlib pyyaml
# !git clone https://github.com/<you>/qm-property-predictor.git
# %cd qm-property-predictor


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from qmprop import load_config
cfg = load_config()
cfg['dataset']


## 1. Data — Ch 2

Download, canonicalize, deduplicate. Watch the log line: the
duplicate count is the reason this step exists.


In [ ]:
import logging; logging.basicConfig(level=logging.INFO, force=True)
from qmprop.data import load_dataset

df = load_dataset(cfg)
target = cfg['dataset']['target_name']
df.head()


In [ ]:
df[target].hist(bins=40, figsize=(6,3))
print(df[target].describe())


## 2. Featurize — Ch 1.2 and 1.7


In [ ]:
from qmprop.features import morgan_matrix, descriptor_matrix

smiles = df['smiles'].tolist()
Xm, morgan_names = morgan_matrix(smiles, **cfg['features']['morgan'])
Xd, desc_names = descriptor_matrix(smiles)

print('fingerprints', Xm.shape)
print('descriptors ', Xd.shape)
print('bit density ', Xm.mean().round(4))   # ECFP is very sparse


## 3. Split — scaffold, not random

The cell below quantifies the inflation. Expect the random split to
look meaningfully better on exactly the same model and data — that
gap is the self-deception a random split buys you.


In [ ]:
import numpy as np
from qmprop.splits import scaffold_split, random_split, murcko_scaffold
from qmprop.models import build_model
from qmprop.evaluate import regression_metrics

X = np.hstack([Xm, Xd])
y = df[target].to_numpy(float)

for name, splitter in [('scaffold', scaffold_split), ('random', random_split)]:
    tr, _, te = splitter(smiles, 0.8, 0.0, 0.2, seed=42)
    m = build_model('random_forest')
    m.fit(X[tr], y[tr])
    s = regression_metrics(y[te], m.predict(X[te]))
    print(f"{name:9} RMSE {s['rmse']:.3f}  R2 {s['r2']:.3f}")


In [ ]:
# How concentrated are the scaffolds?
from collections import Counter
counts = Counter(murcko_scaffold(s) for s in smiles)
print(f'{len(counts)} scaffolds over {len(smiles)} molecules')
print('largest groups:', counts.most_common(5))


## 4. Model comparison — Ch 3 vs Ch 4

The question the book leaves open: does the neural net actually beat
gradient boosting at this dataset size?


In [ ]:
tr, _, te = scaffold_split(smiles, 0.8, 0.0, 0.2, seed=42)

for name in ['ridge', 'random_forest', 'xgboost', 'mlp']:
    m = build_model(name)
    m.fit(X[tr], y[tr])
    s = regression_metrics(y[te], m.predict(X[te]))
    print(f"{name:14} RMSE {s['rmse']:.3f}  MAE {s['mae']:.3f}  R2 {s['r2']:.3f}")


## 5. One quantum calculation — Ch 5 and 6

B3LYP/6-31G* on aspirin. Expect tens of seconds. This is the
single-molecule version of what `scripts/03_run_qm.py` batches.


In [ ]:
from qmprop.qm import qm_descriptors

res = qm_descriptors('CC(=O)Oc1ccccc1C(=O)O', max_heavy_atoms=25)
if res.ok:
    print(f'HOMO   {res.qm_homo_ev:8.3f} eV')
    print(f'LUMO   {res.qm_lumo_ev:8.3f} eV')
    print(f'gap    {res.qm_gap_ev:8.3f} eV')
    print(f'dipole {res.qm_dipole_debye:8.3f} D')
else:
    print('failed:', res.reason)


## 6. Next

```bash
python scripts/03_run_qm.py --limit 10   # smoke test
python scripts/03_run_qm.py              # the real run
python scripts/05_ablation.py            # did QM help?
```
